# Cell 1 — Install/check packages

In [1]:
import sys, subprocess

def ensure(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", pkg])

try:
    import segmentation_models_pytorch as smp
except Exception:
    ensure("segmentation-models-pytorch")
    ensure("timm")

try:
    import matplotlib.pyplot as plt
except Exception:
    ensure("matplotlib")
    import matplotlib.pyplot as plt

import torch
import segmentation_models_pytorch as smp

print("torch:", torch.__version__)
print("smp:", smp.__version__)
print("cuda available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 33.6 MB/s eta 0:00:00
torch: 2.10.0+cu128
smp: 0.5.0
cuda available: True


# Cell 2 — Imports

In [2]:
import os, glob, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader

# Cell 3 — Config

In [3]:
SEED = 7
IMG_SIZE = 448
NUM_WORKERS = 2
BATCH_SIZE = 4

# Risk-controlled thresholds from your Phase-8 policy
THRESHOLD_POLICY = {
    "model1_unet": 0.9,
    "model2_deeplabv3plus": 0.5,
    "model3_unetplusplus": 0.5,
}

# Post-processing params (simple + defensible)
CLOSE_KERNEL = 3
MIN_COMPONENT_AREA = 30
MAX_HOLE_AREA = 30

OUT_DIR = Path("/kaggle/working/output/phase9")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(SEED)

print("Threshold policy:", THRESHOLD_POLICY)
print("Output folder:", OUT_DIR)

Threshold policy: {'model1_unet': 0.9, 'model2_deeplabv3plus': 0.5, 'model3_unetplusplus': 0.5}
Output folder: /kaggle/working/output/phase9


# Cell 4 — Device selection

In [4]:
def choose_device():
    if not torch.cuda.is_available():
        print("CUDA not available -> using CPU")
        return torch.device("cpu")

    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    print("GPU:", name, "| capability:", cap)

    # Avoid the earlier Kaggle P100 crash
    if cap[0] < 7:
        print("⚠️ GPU capability < 7.0 -> using CPU to avoid CUDA crash")
        return torch.device("cpu")

    return torch.device("cuda")

device = choose_device()
print("Using device:", device)

GPU: Tesla P100-PCIE-16GB | capability: (6, 0)
⚠️ GPU capability < 7.0 -> using CPU to avoid CUDA crash
Using device: cpu


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


# Cell 5 — Find inputs

In [5]:
VAL_CSV = glob.glob("/kaggle/input/**/val_split.csv", recursive=True)[0]
val_df = pd.read_csv(VAL_CSV)

print("VAL_CSV:", VAL_CSV)
print("val rows:", len(val_df))
print("val columns:", val_df.columns.tolist())

def pick_first(patterns):
    found = []
    for pat in patterns:
        found.extend(glob.glob(pat, recursive=True))
    found = sorted(set(found))
    return found[0] if found else None

MODEL1_CKPT = pick_first([
    "/kaggle/input/**/*model-1-checkpoints*/*best*.pt",
    "/kaggle/input/**/*unet_efficientnetb3_best*.pt",
])

MODEL2_CKPT = pick_first([
    "/kaggle/input/**/*model-2-checkpoints*/*best*.pt",
    "/kaggle/input/**/*deeplabv3plus*best*.pt",
])

MODEL3_CKPT = pick_first([
    "/kaggle/input/**/*model-3-checkpoints*/*best*.pt",
    "/kaggle/input/**/*unetplusplus*best*.pt",
])

print("\nMODEL1_CKPT:", MODEL1_CKPT)
print("MODEL2_CKPT:", MODEL2_CKPT)
print("MODEL3_CKPT:", MODEL3_CKPT)

assert MODEL1_CKPT is not None
assert MODEL2_CKPT is not None
assert MODEL3_CKPT is not None

VAL_CSV: /kaggle/input/datasets/arman00p/phase-3-csv/val_split.csv
val rows: 1695
val columns: ['stem', 'image_path', 'mask_path', 'source', 'empty_mask', 'has_crack']

MODEL1_CKPT: /kaggle/input/datasets/arman00p/model-1-checkpoints-unet-efficientnet-b3/output/SAVED_MODELS/unet_efficientnetb3_best_ep029_20260219_205815.pt
MODEL2_CKPT: /kaggle/input/datasets/arman00p/model-2-checkpointsdeeplabv3plus-efficientnet-b3/deeplabv3plus_efficientnetb3_pt files/deeplabv3plus_efficientnetb3_best_20260222_102303.pt
MODEL3_CKPT: /kaggle/input/datasets/arman00p/model-3-checkpoints-datasetunet-eff-net-b3/unetplusplus_efficientnetb3_best.pt


# Cell 6 — Helper functions

In [6]:
def extract_state_dict_any(ckpt_obj):
    if isinstance(ckpt_obj, dict) and len(ckpt_obj) > 0:
        if all(isinstance(v, torch.Tensor) for v in ckpt_obj.values()):
            return ckpt_obj

    if isinstance(ckpt_obj, dict):
        for key in ["state_dict", "model_state_dict", "model", "net", "weights"]:
            if key in ckpt_obj and isinstance(ckpt_obj[key], dict):
                sd = ckpt_obj[key]
                if len(sd) > 0 and all(isinstance(v, torch.Tensor) for v in sd.values()):
                    return sd

    best = None
    def walk(obj):
        nonlocal best
        if isinstance(obj, dict):
            if len(obj) > 0 and all(isinstance(v, torch.Tensor) for v in obj.values()):
                if best is None or len(obj) > len(best):
                    best = obj
            for v in obj.values():
                walk(v)
        elif isinstance(obj, (list, tuple)):
            for v in obj:
                walk(v)

    walk(ckpt_obj)
    if best is not None:
        return best

    raise ValueError("Could not locate state_dict inside checkpoint")

def strip_prefixes(sd):
    cleaned = {}
    for k, v in sd.items():
        nk = k
        for pref in ("module.", "model.", "net."):
            if nk.startswith(pref):
                nk = nk[len(pref):]
        cleaned[nk] = v
    return cleaned

def iou_empty_aware(gt, pr, eps=1e-7):
    gt = gt.astype(bool)
    pr = pr.astype(bool)
    inter = np.logical_and(gt, pr).sum()
    union = np.logical_or(gt, pr).sum()
    if union == 0:
        return 1.0
    return float(inter / (union + eps))

def dice_empty_aware(gt, pr, eps=1e-7):
    gt = gt.astype(bool)
    pr = pr.astype(bool)
    inter = np.logical_and(gt, pr).sum()
    denom = gt.sum() + pr.sum()
    if denom == 0:
        return 1.0
    return float((2.0 * inter) / (denom + eps))

# Cell 7 — Dataset

In [7]:
class CrackValDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_size: int, preprocess_fn):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.preprocess_fn = preprocess_fn

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = cv2.imread(str(row["image_path"]), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(row["image_path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA).astype(np.float32)

        m = cv2.imread(str(row["mask_path"]), cv2.IMREAD_GRAYSCALE)
        if m is None:
            raise FileNotFoundError(row["mask_path"])
        m = cv2.resize(m, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        m = (m > 127).astype(np.uint8)

        img = self.preprocess_fn(img)

        img_t = torch.from_numpy(img).permute(2, 0, 1).float()
        m_t = torch.from_numpy(m).unsqueeze(0).float()

        meta = {
            "stem": row.get("stem", ""),
            "source": row.get("source", ""),
            "image_path": str(row["image_path"]),
            "mask_path": str(row["mask_path"]),
            "has_crack": int(row.get("has_crack", int(m.sum() > 0))),
        }
        return img_t, m_t, meta

# Cell 8 — Load all 3 models

In [8]:
def load_smp_model(arch_name, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = strip_prefixes(extract_state_dict_any(ckpt))

    encoder_candidates = ["efficientnet-b3", "timm-efficientnet-b3"]
    best = None

    for enc in encoder_candidates:
        if arch_name == "Unet":
            m = smp.Unet(encoder_name=enc, encoder_weights=None, in_channels=3, classes=1, activation=None)
        elif arch_name == "DeepLabV3Plus":
            m = smp.DeepLabV3Plus(encoder_name=enc, encoder_weights=None, in_channels=3, classes=1, activation=None)
        elif arch_name == "UnetPlusPlus":
            m = smp.UnetPlusPlus(encoder_name=enc, encoder_weights=None, in_channels=3, classes=1, activation=None)
        else:
            raise ValueError(f"Unknown arch: {arch_name}")

        missing, unexpected = m.load_state_dict(sd, strict=False)
        score = len(missing) + len(unexpected)
        print(f"[{arch_name} | {enc}] missing={len(missing)} unexpected={len(unexpected)} score={score}")

        if best is None or score < best["score"]:
            best = {"model": m, "encoder": enc, "score": score}

    if best["score"] > 150:
        raise RuntimeError(f"{arch_name} checkpoint load looks wrong. Score={best['score']}")

    return best["model"].to(device).eval(), best["encoder"]

loaded_models = {
    "model1_unet": {"arch": "Unet", "ckpt": MODEL1_CKPT},
    "model2_deeplabv3plus": {"arch": "DeepLabV3Plus", "ckpt": MODEL2_CKPT},
    "model3_unetplusplus": {"arch": "UnetPlusPlus", "ckpt": MODEL3_CKPT},
}

for name, info in loaded_models.items():
    print(f"\nLoading {name} ...")
    model, enc = load_smp_model(info["arch"], info["ckpt"])
    info["model"] = model
    info["encoder"] = enc

print("\n All models loaded")


Loading model1_unet ...
[Unet | efficientnet-b3] missing=494 unexpected=572 score=1066
[Unet | timm-efficientnet-b3] missing=0 unexpected=0 score=0

Loading model2_deeplabv3plus ...
[DeepLabV3Plus | efficientnet-b3] missing=0 unexpected=0 score=0
[DeepLabV3Plus | timm-efficientnet-b3] missing=494 unexpected=572 score=1066

Loading model3_unetplusplus ...
[UnetPlusPlus | efficientnet-b3] missing=0 unexpected=0 score=0
[UnetPlusPlus | timm-efficientnet-b3] missing=494 unexpected=572 score=1066

 All models loaded


# Cell 9 — Post-processing functions

In [9]:
def remove_small_components(mask, min_area=30):
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    out = np.zeros_like(mask, dtype=np.uint8)
    for k in range(1, n):
        area = stats[k, cv2.CC_STAT_AREA]
        if area >= min_area:
            out[labels == k] = 1
    return out

def fill_small_holes(mask, max_hole_area=30):
    # invert: holes become components
    inv = 1 - mask.astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(inv, connectivity=8)
    H, W = mask.shape
    out = mask.copy().astype(np.uint8)

    for k in range(1, n):
        area = stats[k, cv2.CC_STAT_AREA]
        ys, xs = np.where(labels == k)
        touches_border = (xs == 0).any() or (xs == W-1).any() or (ys == 0).any() or (ys == H-1).any()
        if (not touches_border) and area <= max_hole_area:
            out[labels == k] = 1
    return out

def postprocess_mask(mask, close_kernel=3, min_area=30, max_hole_area=30):
    m = mask.astype(np.uint8)

    # closing: helps close small gaps / holes
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel, close_kernel))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel)

    # remove tiny blobs
    m = remove_small_components(m, min_area=min_area)

    # fill tiny holes
    m = fill_small_holes(m, max_hole_area=max_hole_area)

    return m.astype(np.uint8)

# Cell 10 — Evaluate raw vs postprocessed

In [10]:
def evaluate_model_raw_vs_post(model_name, model, encoder_name, threshold):
    preprocess_fn = smp.encoders.get_preprocessing_fn(encoder_name, pretrained="imagenet")
    ds = CrackValDataset(val_df, img_size=IMG_SIZE, preprocess_fn=preprocess_fn)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    per_image_rows = []

    for imgs, msks, metas in dl:
        imgs = imgs.to(device)
        with torch.inference_mode():
            prob = torch.sigmoid(model(imgs)).cpu().numpy()[:,0]

        gt_batch = msks.numpy()[:,0].astype(np.uint8)

        for b in range(prob.shape[0]):
            gt = gt_batch[b]
            raw = (prob[b] >= threshold).astype(np.uint8)
            post = postprocess_mask(
                raw,
                close_kernel=CLOSE_KERNEL,
                min_area=MIN_COMPONENT_AREA,
                max_hole_area=MAX_HOLE_AREA
            )

            has_crack_gt = int(gt.sum() > 0)

            per_image_rows.append({
                "model_name": model_name,
                "stem": metas["stem"][b],
                "source": metas["source"][b],
                "image_path": metas["image_path"][b],
                "mask_path": metas["mask_path"][b],
                "has_crack_gt": has_crack_gt,
                "raw_pred_any": int(raw.sum() > 0),
                "post_pred_any": int(post.sum() > 0),
                "gt_pixels": int(gt.sum()),
                "raw_pixels": int(raw.sum()),
                "post_pixels": int(post.sum()),
                "raw_iou": iou_empty_aware(gt, raw),
                "post_iou": iou_empty_aware(gt, post),
                "raw_dice": dice_empty_aware(gt, raw),
                "post_dice": dice_empty_aware(gt, post),
            })

    return pd.DataFrame(per_image_rows)

# Cell 11 — Run all 3 models and save reports

In [11]:
all_per_image = []

for model_name, info in loaded_models.items():
    print(f"\nRunning raw vs post for {model_name} ...")
    df_model = evaluate_model_raw_vs_post(
        model_name=model_name,
        model=info["model"],
        encoder_name=info["encoder"],
        threshold=THRESHOLD_POLICY[model_name]
    )
    all_per_image.append(df_model)

per_image_df = pd.concat(all_per_image, ignore_index=True)
per_image_path = OUT_DIR / "phase9_postproc_per_image.csv"
per_image_df.to_csv(per_image_path, index=False)

summary_rows = []
for model_name in per_image_df["model_name"].unique():
    sub = per_image_df[per_image_df["model_name"] == model_name]
    crack = sub[sub["has_crack_gt"] == 1]
    no_crack = sub[sub["has_crack_gt"] == 0]

    summary_rows.append({
        "model_name": model_name,
        "threshold_used": THRESHOLD_POLICY[model_name],
        "raw_mean_iou_all": sub["raw_iou"].mean(),
        "post_mean_iou_all": sub["post_iou"].mean(),
        "raw_mean_iou_crack_only": crack["raw_iou"].mean(),
        "post_mean_iou_crack_only": crack["post_iou"].mean(),
        "raw_mean_dice_crack_only": crack["raw_dice"].mean(),
        "post_mean_dice_crack_only": crack["post_dice"].mean(),
        "raw_no_crack_fp_rate": no_crack["raw_pred_any"].mean(),
        "post_no_crack_fp_rate": no_crack["post_pred_any"].mean(),
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = OUT_DIR / "phase9_postproc_report.csv"
summary_df.to_csv(summary_path, index=False)

print(" Saved:", per_image_path)
print(" Saved:", summary_path)
summary_df


Running raw vs post for model1_unet ...


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]


Running raw vs post for model2_deeplabv3plus ...


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]


Running raw vs post for model3_unetplusplus ...
 Saved: /kaggle/working/output/phase9/phase9_postproc_per_image.csv
 Saved: /kaggle/working/output/phase9/phase9_postproc_report.csv


,model_name,threshold_used,raw_mean_iou_all,post_mean_iou_all,raw_mean_iou_crack_only,post_mean_iou_crack_only,raw_mean_dice_crack_only,post_mean_dice_crack_only,raw_no_crack_fp_rate,post_no_crack_fp_rate
0,model1_unet,0.9,0.621760,0.621644,0.568642,0.568508,0.710454,0.710253,0.018349,0.018349
1,model2_deeplabv3plus,0.5,0.627161,0.627246,0.572808,0.572229,0.713919,0.712994,0.004587,0.000000
2,model3_unetplusplus,0.5,0.631079,0.632109,0.580690,0.580518,0.721545,0.721360,0.027523,0.018349


# Cell 12 — Plot raw vs post delta

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))

x = np.arange(len(summary_df))
labels = summary_df["model_name"].tolist()

# crack IoU
axes[0].bar(x - 0.15, summary_df["raw_mean_iou_crack_only"], width=0.3, label="raw")
axes[0].bar(x + 0.15, summary_df["post_mean_iou_crack_only"], width=0.3, label="post")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=15)
axes[0].set_title("Raw vs Postprocessed — Crack-only IoU")
axes[0].legend()

# no-crack FP rate
axes[1].bar(x - 0.15, summary_df["raw_no_crack_fp_rate"], width=0.3, label="raw")
axes[1].bar(x + 0.15, summary_df["post_no_crack_fp_rate"], width=0.3, label="post")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=15)
axes[1].set_title("Raw vs Postprocessed — No-crack FP rate")
axes[1].legend()

plt.tight_layout()
plot_path = OUT_DIR / "phase9_postproc_delta_plot.png"
plt.savefig(plot_path, dpi=220)
plt.close()

print(" Saved:", plot_path)

 Saved: /kaggle/working/output/phase9/phase9_postproc_delta_plot.png


# Cell 13 — Example figure - extra

In [13]:
def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return img

def read_mask(path):
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    return (m > 127).astype(np.uint8)

def overlay_mask(img, mask, color=(255,0,0), alpha=0.45):
    out = img.copy().astype(np.float32)
    m = mask.astype(bool)
    out[m] = out[m] * (1 - alpha) + np.array(color, dtype=np.float32) * alpha
    return out.astype(np.uint8)

fig_rows = []

for model_name in per_image_df["model_name"].unique():
    sub = per_image_df[per_image_df["model_name"] == model_name].copy()
    sub = sub[sub["has_crack_gt"] == 1].copy()
    sub["improve"] = sub["post_iou"] - sub["raw_iou"]
    top = sub.sort_values("improve", ascending=False).head(3)
    fig_rows.append((model_name, top))

nrows = sum(len(t) for _, t in fig_rows)
fig, axes = plt.subplots(nrows, 4, figsize=(12, 3*nrows))
if nrows == 1:
    axes = np.expand_dims(axes, 0)

r = 0
for model_name, top_df in fig_rows:
    info = loaded_models[model_name]
    preprocess_fn = smp.encoders.get_preprocessing_fn(info["encoder"], pretrained="imagenet")

    for _, row in top_df.iterrows():
        img = read_rgb(row["image_path"])
        gt = read_mask(row["mask_path"])

        img_in = preprocess_fn(img.astype(np.float32))
        x = torch.from_numpy(img_in).permute(2,0,1).unsqueeze(0).float().to(device)

        with torch.inference_mode():
            prob = torch.sigmoid(info["model"](x)).cpu().numpy()[0,0]

        raw = (prob >= THRESHOLD_POLICY[model_name]).astype(np.uint8)
        post = postprocess_mask(raw, close_kernel=CLOSE_KERNEL, min_area=MIN_COMPONENT_AREA, max_hole_area=MAX_HOLE_AREA)

        panels = [
            (f"{model_name}\nImage", img),
            ("GT", gt * 255),
            (f"Raw overlay\nIoU={row['raw_iou']:.3f}", overlay_mask(img, raw)),
            (f"Post overlay\nIoU={row['post_iou']:.3f}", overlay_mask(img, post)),
        ]

        for c, (title, im) in enumerate(panels):
            ax = axes[r, c]
            ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
            ax.set_title(title, fontsize=9)
            ax.axis("off")
        r += 1

plt.tight_layout()
examples_path = OUT_DIR / "phase9_postproc_examples.png"
plt.savefig(examples_path, dpi=220)
plt.close()

print(" Saved:", examples_path)

 Saved: /kaggle/working/output/phase9/phase9_postproc_examples.png
